# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedshereef1/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
# ============================================================
# 1. DISTRIBUTIONS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Load the FlyRank starter dataset
# ------------------------------------------------------------

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))


# ------------------------------------------------------------
# Check the signals we will audit
# ------------------------------------------------------------

signals = [
    "days_since_last_update",
    "search_volume",
    "ctr",
    "avg_position",
]

print("\nMissing values:")
display(
    df[signals].isna().sum().rename("missing_n").to_frame()
)


# ------------------------------------------------------------
# Descriptive statistics
# ------------------------------------------------------------

print("\nSignal distributions:")
display(
    df[signals].describe()
)


# ------------------------------------------------------------
# Heavy-tail check
# ------------------------------------------------------------

print("\nSelected percentiles:")

percentiles = [0.50, 0.75, 0.90, 0.95, 0.99]

distribution_table = pd.DataFrame({
    "days_since_last_update": df["days_since_last_update"].quantile(percentiles),
    "search_volume": df["search_volume"].quantile(percentiles),
    "ctr": df["ctr"].quantile(percentiles),
    "avg_position": df["avg_position"].replace(0, np.nan).quantile(percentiles),
})

distribution_table.index = [
    "50th percentile",
    "75th percentile",
    "90th percentile",
    "95th percentile",
    "99th percentile",
]

display(distribution_table)


# ------------------------------------------------------------
# Notes
# ------------------------------------------------------------

print("\nDistribution notes:")
print("- Search volume is expected to be heavy-tailed.")
print("- CTR is a percentage stored as a ×100 value.")
print("- avg_position = 0 means no position data and is excluded")
print("  from position calculations.")
print("- Bucketed medians will be used instead of raw Pearson")
print("  correlations for the signal tests.")

Dataset shape: (30000, 44)
Number of columns: 44

Missing values:


,missing_n
days_since_last_update,0
search_volume,2468
ctr,0
avg_position,0



Signal distributions:


,days_since_last_update,search_volume,ctr,avg_position
count,30000.000000,27532.000000,30000.000000,30000.00000
mean,46.098300,158.882391,0.510733,16.34238
std,42.078709,1518.270825,3.279162,15.21679
min,1.000000,0.000000,0.000000,0.00000
25%,20.000000,0.000000,0.000000,6.20000
50%,20.000000,10.000000,0.070000,10.80000
75%,104.000000,20.000000,0.290000,22.30000
max,373.000000,74000.000000,100.000000,245.00000



Selected percentiles:


,days_since_last_update,search_volume,ctr,avg_position
50th percentile,20.0,10.0,0.07,11.4
75th percentile,104.0,20.0,0.29,22.9
90th percentile,104.0,110.0,0.65,37.5
95th percentile,104.0,390.0,1.09,48.8
99th percentile,106.0,2900.0,8.33,70.5



Distribution notes:
- Search volume is expected to be heavy-tailed.
- CTR is a percentage stored as a ×100 value.
- avg_position = 0 means no position data and is excluded
  from position calculations.
- Bucketed medians will be used instead of raw Pearson
  correlations for the signal tests.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [5]:
# ============================================================
# 2. SIGNAL TEST #1 / #2 / #3
# ============================================================

# We use the current observed trend_direction only as an
# audit outcome, not as a model feature.
#
# IMPORTANT:
# trend_direction is derived from the current 30-day comparison
# window, so this is a directional signal audit, NOT a
# future-looking prediction.


# ------------------------------------------------------------
# Create a decline indicator for AUDIT ONLY
# ------------------------------------------------------------

df["declining_audit"] = (
    df["trend_direction"] == "down"
).astype(int)

print(
    "Declining audit rate:",
    round(df["declining_audit"].mean() * 100, 2),
    "%"
)


# ============================================================
# TEST 1 — STALENESS
# Claim:
# Older content is more likely to show declining performance.
# ============================================================

staleness_bins = [-1, 30, 60, 90, 120, 180, 365, np.inf]

staleness_labels = [
    "0-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
    ">365",
]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=staleness_bins,
    labels=staleness_labels
)

staleness_test = (
    df.groupby("staleness_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        decline_rate=("declining_audit", "mean"),
        median_search_volume=("search_volume", "median"),
    )
    .reset_index()
)

staleness_test["decline_rate_pct"] = (
    staleness_test["decline_rate"] * 100
)

print("\n" + "=" * 70)
print("SIGNAL TEST #1 — STALENESS")
print("=" * 70)

display(
    staleness_test[
        [
            "staleness_bucket",
            "n",
            "decline_rate_pct",
            "median_search_volume",
        ]
    ]
)


# Verdict based only on buckets with n >= 50
valid_staleness = staleness_test[
    staleness_test["n"] >= 50
]

if len(valid_staleness) < 2:
    staleness_verdict = "FALSE"
else:
    first_rate = valid_staleness["decline_rate_pct"].iloc[0]
    last_rate = valid_staleness["decline_rate_pct"].iloc[-1]

    if last_rate > first_rate + 5:
        staleness_verdict = "CONFIRMED"
    elif last_rate < first_rate - 5:
        staleness_verdict = "OPPOSITE"
    else:
        staleness_verdict = "MIXED"

print("Staleness verdict:", staleness_verdict)


# ============================================================
# TEST 2 — SEARCH VOLUME
# Claim:
# Higher search volume represents greater observable demand.
# ============================================================

volume_bins = [-np.inf, 0, 10, 50, 100, 500, 1000, np.inf]

volume_labels = [
    "0",
    "1-10",
    "11-50",
    "51-100",
    "101-500",
    "501-1000",
    ">1000",
]

df["volume_bucket"] = pd.cut(
    df["search_volume"],
    bins=volume_bins,
    labels=volume_labels
)

volume_test = (
    df.groupby("volume_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_clicks=("clicks_90d", "median"),
        decline_rate=("declining_audit", "mean"),
    )
    .reset_index()
)

volume_test["decline_rate_pct"] = (
    volume_test["decline_rate"] * 100
)

print("\n" + "=" * 70)
print("SIGNAL TEST #2 — SEARCH VOLUME")
print("=" * 70)

display(
    volume_test[
        [
            "volume_bucket",
            "n",
            "median_impressions",
            "median_clicks",
            "decline_rate_pct",
        ]
    ]
)


# Verdict based on observed opportunity pattern
valid_volume = volume_test[
    volume_test["n"] >= 50
]

if len(valid_volume) < 2:
    volume_verdict = "FALSE"
else:
    low_volume_impressions = valid_volume["median_impressions"].iloc[0]
    high_volume_impressions = valid_volume["median_impressions"].iloc[-1]

    if high_volume_impressions > low_volume_impressions * 1.5:
        volume_verdict = "CONFIRMED"
    elif high_volume_impressions < low_volume_impressions / 1.5:
        volume_verdict = "OPPOSITE"
    else:
        volume_verdict = "MIXED"

print("Search-volume verdict:", volume_verdict)


# ============================================================
# TEST 3 — CTR VS POSITION
# ============================================================

# Claim:
# CTR should generally be higher for pages with better
# search positions, among pages with meaningful impressions.

position_df = df.copy()

# 0 means no usable position data
position_df.loc[
    position_df["avg_position"] <= 0,
    "avg_position"
] = np.nan

# Require meaningful search exposure before comparing CTR.
# This reduces noise from pages with very little traffic.
position_df = position_df[
    position_df["impressions_90d"] >= 100
].copy()

position_bins = [
    0,
    3,
    10,
    20,
    50,
    np.inf,
]

position_labels = [
    "top_3",
    "page_1",
    "striking",
    "page_3_5",
    "deep",
]

position_df["position_bucket"] = pd.cut(
    position_df["avg_position"],
    bins=position_bins,
    labels=position_labels
)

position_test = (
    position_df.groupby("position_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        median_ctr=("ctr", "median"),
        median_impressions=("impressions_90d", "median"),
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("SIGNAL TEST #3 — CTR VS POSITION")
print("=" * 70)

display(position_test)


# ------------------------------------------------------------
# Verdict
# ------------------------------------------------------------

valid_position = position_test[
    position_test["n"] >= 50
].copy()

if len(valid_position) < 2:
    position_verdict = "FALSE"

else:
    ctr_values = valid_position["median_ctr"].to_numpy()

    # Check whether CTR generally decreases as position worsens.
    decreasing_steps = np.sum(np.diff(ctr_values) < 0)
    increasing_steps = np.sum(np.diff(ctr_values) > 0)

    if decreasing_steps >= 3:
        position_verdict = "CONFIRMED"

    elif increasing_steps >= 3:
        position_verdict = "OPPOSITE"

    else:
        position_verdict = "MIXED"

print("CTR-vs-position verdict:", position_verdict)
# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SIGNAL AUDIT SUMMARY")
print("=" * 70)

print(f"1. Staleness:       {staleness_verdict}")
print(f"2. Search volume:   {volume_verdict}")
print(f"3. CTR vs position: {position_verdict}")

Declining audit rate: 54.21 %

SIGNAL TEST #1 — STALENESS


,staleness_bucket,n,decline_rate_pct,median_search_volume
0,0-30,20480,51.137695,10.0
1,31-60,128,58.593750,10.0
2,61-90,47,59.574468,10.0
3,91-120,9115,61.305540,10.0
4,121-180,56,28.571429,20.0
5,181-365,169,46.745562,0.0
6,>365,5,60.000000,0.0


Staleness verdict: MIXED

SIGNAL TEST #2 — SEARCH VOLUME


,volume_bucket,n,median_impressions,median_clicks,decline_rate_pct
0,0,11081,998.0,1.0,63.243390
1,1-10,7311,834.0,1.0,52.660375
2,11-50,4989,935.0,1.0,52.495490
3,51-100,1102,842.5,1.0,50.090744
4,101-500,2021,796.0,1.0,51.360713
5,501-1000,468,826.0,0.0,44.658120
6,>1000,560,845.5,0.0,44.464286


Search-volume verdict: MIXED

SIGNAL TEST #3 — CTR VS POSITION


,position_bucket,n,median_ctr,median_impressions
0,top_3,555,0.19,3047.0
1,page_1,8660,0.23,2940.0
2,striking,5876,0.15,1377.0
3,page_3_5,6037,0.06,1209.0
4,deep,878,0.00,427.0


CTR-vs-position verdict: CONFIRMED

SIGNAL AUDIT SUMMARY
1. Staleness:       MIXED
2. Search volume:   MIXED
3. CTR vs position: CONFIRMED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
# ============================================================
# 3. THE FLAG-LINKED TEST
# ============================================================

# FlyRank's refresh logic uses content freshness/staleness as an
# important observable signal.
#
# Product flags themselves are NOT included in this dataset.
# Therefore, we test the underlying observable assumption:
#
# "Older content should show more evidence of decline."


# ------------------------------------------------------------
# Compare recent vs stale content
# ------------------------------------------------------------

df["staleness_group"] = np.where(
    df["days_since_last_update"] > 90,
    "stale_90_plus",
    "recent_90_or_less",
)

flag_linked_test = (
    df.groupby("staleness_group")
    .agg(
        n=("content_id", "size"),
        decline_rate=("declining_audit", "mean"),
        median_impressions=("impressions_90d", "median"),
        median_clicks=("clicks_90d", "median"),
    )
    .reset_index()
)

flag_linked_test["decline_rate_pct"] = (
    flag_linked_test["decline_rate"] * 100
)

print("=" * 70)
print("FLAG-LINKED TEST — STALENESS / REFRESH")
print("=" * 70)

display(
    flag_linked_test[
        [
            "staleness_group",
            "n",
            "decline_rate_pct",
            "median_impressions",
            "median_clicks",
        ]
    ]
)


# ------------------------------------------------------------
# Verdict
# ------------------------------------------------------------

recent = flag_linked_test[
    flag_linked_test["staleness_group"] == "recent_90_or_less"
]

stale = flag_linked_test[
    flag_linked_test["staleness_group"] == "stale_90_plus"
]

if len(recent) == 1 and len(stale) == 1:
    recent_rate = recent["decline_rate_pct"].iloc[0]
    stale_rate = stale["decline_rate_pct"].iloc[0]

    recent_n = recent["n"].iloc[0]
    stale_n = stale["n"].iloc[0]

    if recent_n >= 50 and stale_n >= 50:

        if stale_rate > recent_rate + 5:
            flag_verdict = "CONFIRMED"

        elif stale_rate < recent_rate - 5:
            flag_verdict = "OPPOSITE"

        else:
            flag_verdict = "MIXED"

    else:
        flag_verdict = "FALSE"

else:
    flag_verdict = "FALSE"


print("\nFlag-linked verdict:", flag_verdict)


# ------------------------------------------------------------
# Practical interpretation
# ------------------------------------------------------------

print("\nInterpretation:")

if flag_verdict == "CONFIRMED":
    print(
        "The observed data supports the directional assumption that "
        "staler content shows a higher decline rate. This supports "
        "using freshness as a refresh-prioritization signal."
    )

elif flag_verdict == "OPPOSITE":
    print(
        "The observed data moves in the opposite direction. "
        "Staleness should therefore not be treated as evidence "
        "that a refresh is needed by itself."
    )

elif flag_verdict == "MIXED":
    print(
        "The observed data does not show a strong enough difference "
        "to confidently support or reject the refresh assumption. "
        "Staleness should therefore be treated as a directional "
        "signal rather than a decisive rule."
    )

else:
    print(
        "There is not enough usable data in the comparison groups "
        "to make a reliable verdict."
    )

FLAG-LINKED TEST — STALENESS / REFRESH


,staleness_group,n,decline_rate_pct,median_impressions,median_clicks
0,recent_90_or_less,20655,51.203099,472.0,1.0
1,stale_90_plus,9345,60.845372,1621.0,2.0



Flag-linked verdict: CONFIRMED

Interpretation:
The observed data supports the directional assumption that staler content shows a higher decline rate. This supports using freshness as a refresh-prioritization signal.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [4]:
# ============================================================
# 4. WHAT THIS MEANS IN PRACTICE
# ============================================================

print("Practical takeaway")
print("=" * 70)

print(
    "The signal audit shows that staleness, search volume, and "
    "CTR-versus-position contain observable variation that can "
    "help prioritize content for review."
)

print(
    "However, the verdicts are directional rather than causal. "
    "A signal should not be treated as proof that a page needs "
    "a specific action."
)

print(
    "For the baseline, the strongest use is to combine simple "
    "observable signals into a transparent review queue and "
    "then validate whether a later model improves on that rule."
)

Practical takeaway
The signal audit shows that staleness, search volume, and CTR-versus-position contain observable variation that can help prioritize content for review.
However, the verdicts are directional rather than causal. A signal should not be treated as proof that a page needs a specific action.
For the baseline, the strongest use is to combine simple observable signals into a transparent review queue and then validate whether a later model improves on that rule.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.